# Multi-Step Prompt Chaining

In this notebook, we'll explore how to do **prompt chaining** to handle complex tasks using OpenAI models.  
We'll cover three patterns:
- Sequential chaining
- Conditional chaining
- Looping chaining



## Setup
Install OpenAI Python SDK and set your API key.

In [1]:
!pip install -qU openai

In [2]:
import openai
from openai import OpenAI
from google.colab import userdata
import os

# Make sure you set your OPENAI_API_KEY as environment variable:
# (In Colab: click on the left, go to "Variables", and add OPENAI_API_KEY)
# Retrieve the API key and store it to variable
openai.api_key = userdata.get("OPENAI_API_KEY")
client = OpenAI(api_key=openai.api_key)


First, we create a helper function called `get_completion` that sends a prompt to the OpenAI chat model and returns the text response. This helps keep our prompt chaining code clean, simple, and reusable.

In [3]:
def get_completion(prompt, model="gpt-4o-mini"):
    """
    A helper function to get completion from OpenAI chat models.
    """
    messages = [{"role": "user", "content": prompt}]
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=0
    )
    return response.choices[0].message.content

## Part 1: Sequential Chaining

In **sequential chaining**, we break a complex task into smaller subtasks.  
Each step's output becomes the next step's input.

Let's analyze a paragraph about a family and do multiple steps:

**Paragraph:**
> "John is 23 years old, his sister Anna just turned 19 last month, and their father David is now 50. Their grandmother, who moved in recently, is 75 years old."

We'll chain four steps:
1. Extract all ages into a list
2. Identify the oldest person mentioned
3. Generate a short summary sentence about the family’s age range


In [4]:
paragraph = """
John is 23 years old, his sister Anna just turned 19 last month, and their father David is now 50.
Their grandmother, who moved in recently, is 75 years old.
"""

# Step 1: Extract ages as list
prompt1 = f"Extract all ages mentioned in this paragraph: '''{paragraph}''' \n make the output format as name: age"
ages_list = get_completion(prompt1)
print("Step 1 - Extracted Ages:")
print(ages_list)

# Step 2: Identify oldest person
prompt2 = f"Based on this {ages_list}, who is the oldest person mentioned?"
oldest_person = get_completion(prompt2)
print("\n Step 2 - Oldest Person:")
print(oldest_person)

# Step 3: Generate summary sentence
prompt3 = f"Write a short summary sentence describing the family and who is the oldest based on this information: {oldest_person} and based on the paragraph: {paragraph}"
summary_sentence = get_completion(prompt3)
print("\nStep 3 - Summary Sentence:")
print(summary_sentence)


Step 1 - Extracted Ages:
John: 23  
Anna: 19  
David: 50  
Grandmother: 75  

 Step 2 - Oldest Person:
The oldest person mentioned is the Grandmother, who is 75 years old.

Step 3 - Summary Sentence:
The family consists of Grandmother (75), Father David (50), son John (23), and daughter Anna (19), with the Grandmother being the oldest member.


## Part 2: Conditional Chaining

In **conditional chaining**, we decide the next prompt based on a previous result.

In this example, we build a **conditional prompt chain** where the *next step* depends on the *sentiment* of the text.

Here’s how it works:
- **Step 1:** Ask the model to analyze the current state of renewable energy adoption globally.
- **Step 2:** Check the sentiment of that analysis (positive, negative, or neutral).
- **Step 3:** Based on the sentiment, send a different follow-up prompt:
  - Positive → ask for opportunities
  - Negative → ask for solutions
  - Neutral → ask for key focus areas
- **Step 4:** Return the final recommendations tailored to the sentiment.

In [5]:
def analyze_sentiment(text):
    prompt = f"Analyze the sentiment of the following text and respond with only one word - 'positive', 'negative', or 'neutral': {text}"
    sentiment = get_completion(prompt)

    print("\nStep 2 - Analyze Sentiment")
    print(sentiment)

    return sentiment.strip().lower()

def conditional_prompt_chain(result):
    if result is None:
        return "Initial prompt failed."
    sentiment = analyze_sentiment(result)

    print(f"Sentiment: {sentiment}\n")
    if sentiment == 'positive':
        follow_up = "Given this positive outlook, what are three potential opportunities we can explore?"
    elif sentiment == 'negative':
        follow_up = "Considering these challenges, what are three possible solutions we can implement?"
    else:  # neutral
        follow_up = "Based on this balanced view, what are three key areas we should focus on for a comprehensive approach?"

    final_result = get_completion(f"{follow_up}\n\nContext: {result}")
    return final_result

# Example usage
initial_prompt = "Analyze the current state of renewable energy adoption globally."
result = get_completion(initial_prompt)
print("Step 1 - Analyze Current State of Renewable Energy:")
print(result)

final_result = conditional_prompt_chain(result)
print("Step 3 - Solution:")
print(final_result)

Step 1 - Analyze Current State of Renewable Energy:
As of October 2023, the global landscape of renewable energy adoption has seen significant advancements, driven by technological innovations, policy frameworks, and increasing awareness of climate change. Here are some key trends and insights regarding the current state of renewable energy:

### 1. **Growth in Renewable Energy Capacity**
- **Installed Capacity**: The total installed capacity of renewable energy sources, particularly solar and wind, has continued to grow rapidly. According to the International Renewable Energy Agency (IRENA), global renewable energy capacity surpassed 3,000 GW, with solar and wind accounting for a substantial portion of this growth.
- **Solar Energy**: Solar photovoltaic (PV) installations have seen exponential growth, with countries like China, the United States, and India leading in capacity additions. The cost of solar technology has decreased significantly, making it more accessible.
- **Wind Energ

## Part 3: Looping Chaining

In **looping chaining**, we repeat prompts until a condition is met.

In this example, we want to **extract product features one by one** from a paragraph.
Since we don’t know how many features there are in advance, we loop:
- At each step, we ask the LLM:  
  > “Give me ONE new feature not already listed. If there are none left, reply with 'DONE'.”
- We keep track of the features we’ve already extracted and include them in the prompt.
- When the model finally says “DONE”, we stop the loop.

In [7]:
# Original paragraph
product_paragraph = """
This smartphone comes with a 48MP rear camera, a 12MP front camera, 128GB of storage, a long-lasting 4000mAh battery,
and supports fast charging. It also has face unlock and an in-display fingerprint sensor.
"""

# Initialize empty list to collect features
features = []

# Initial prompt template
base_prompt = """
You are extracting product features from a paragraph.
List ONE new product feature you haven't already listed.
If there really are no more new features, reply exactly with 'DONE'.

Paragraph:
'''{paragraph}'''

Features already listed so far: {listed}

Remember: there ARE features in the paragraph, so find them one by one.
Return ONLY the new feature text (not numbered, not extra words), or 'DONE'.
"""

for i in range(10):  # Safety cap: max 10 iterations
    # Create prompt with current list
    listed = "; ".join(features) if features else "None"
    prompt = base_prompt.format(paragraph=product_paragraph, listed=listed)

    # Get completion
    feature = get_completion(prompt)
    print(f"🔁 Loop Step {i+1} - Extracted: {feature}")

    if "DONE" in feature.upper():
        break
    else:
        features.append(feature.strip())

print("\n📦 Final list of features extracted:")
for idx, f in enumerate(features, 1):
    print(f"{idx}. {f}")

🔁 Loop Step 1 - Extracted: 48MP rear camera
🔁 Loop Step 2 - Extracted: 12MP front camera
🔁 Loop Step 3 - Extracted: 128GB of storage
🔁 Loop Step 4 - Extracted: 4000mAh battery
🔁 Loop Step 5 - Extracted: fast charging
🔁 Loop Step 6 - Extracted: face unlock
🔁 Loop Step 7 - Extracted: in-display fingerprint sensor
🔁 Loop Step 8 - Extracted: DONE

📦 Final list of features extracted:
1. 48MP rear camera
2. 12MP front camera
3. 128GB of storage
4. 4000mAh battery
5. fast charging
6. face unlock
7. in-display fingerprint sensor


## Summary

So far in this notebook, we have:

1.  Installed the necessary library (`openai`).
2.  Set up the OpenAI API key.
3.  Created a helper function `get_completion` to easily interact with the OpenAI chat model.
4.  Explored **Sequential Chaining** by breaking down a task into steps and feeding the output of one step into the next.
5.  Explored **Conditional Chaining** by using the sentiment of text to decide the next action.
6.  Explored **Looping Chaining** by repeatedly asking the model for more information until a condition is met.